# CachePilot：Colab 单 GPU Phase 1 完整验证

目标：得到 TODO 2.3 的**渐进压测、必要对照、故障验证、Phase 1 出口**四项结论。顺序执行所有单元，最后下载原始证据、`conclusions.json` 和 `report.md`。失败也保留数据，不自动勾选 TODO。

先选择「运行时 → 更改运行时类型 → GPU」，将包含本次修改和 .git 的完整仓库放入 Drive。全流程使用独立 Python 3.13.15 + 真实 TorchExecutor + 固定 Qwen revision；T4 选 float16，支持 BF16 的卡选 bfloat16。同一矩阵不切换硬件/dtype。下载和完整扫描可能需要数十分钟至数小时。

**语义边界**：Torch 当前先完成整段 generate 再交付 token，TTFT/TPOT 是 Gateway 交付指标，不能解释为逐 token GPU decode 性能；物理 prefix 命中不可观测。static/continuous 对照为 N/A。vLLM 接入检查使用另一个 `colab_acceptance.ipynb`，不混入本矩阵。

## 1. 工作目录、代码快照与日志

修改 SOURCE 后执行；代码复制到 /content，避免 Drive FUSE 安装问题。每次创建独立会话，禁止混合不同会话的重复次数。不在 notebook 填写任何凭据。后续任一步失败，先修复或直接执行最后的归档单元。

In [1]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import atexit, hashlib, json, os, re, shutil, subprocess, sys, time, urllib.request

if "stop_server" in globals():
    stop_server()
    atexit.unregister(stop_server)

drive.mount("/content/drive")
SOURCE = Path("/content/drive/MyDrive/CachePilot")
SESSION = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + os.urandom(3).hex()
WORKDIR = Path("/content") / ("CachePilot-" + SESSION)
OUT = Path("/content/cachepilot-phase1") / SESSION
BACKUP = Path("/content/drive/MyDrive/CachePilot-evidence") / SESSION
assert (SOURCE / ".git").exists(), "需要完整仓库以记录 commit"
assert (SOURCE / "benchmarks/colab_phase1.py").exists(), "请先同步新增脚本"
for name in ("manifest.json", "trace.jsonl", "requests.jsonl"):
    assert (SOURCE / "tests/fixtures/diagnostics" / name).is_file(), "请同步 tests/fixtures/diagnostics 全部文件"
shutil.copytree(SOURCE, WORKDIR, ignore=shutil.ignore_patterns(
    "runs", ".venv", "venv", "__pycache__", ".pytest_cache", ".ruff_cache", ".idea"))
OUT.mkdir(parents=True)
BACKUP.mkdir(parents=True)
os.chdir(WORKDIR)
os.environ["PYTHONPATH"] = str(WORKDIR)

def save(name, value):
    target = OUT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n")

def checkpoint():
    shutil.copytree(OUT, BACKUP, dirs_exist_ok=True)

def redact(text):
    text = re.sub(r"hf_[A-Za-z0-9]+", "[REDACTED]", text)
    return re.sub(r"(?i)bearer\s+\S+", "Bearer [REDACTED]", text)

def command(argv, label, timeout=3600, required=True):
    try:
        result = subprocess.run(list(map(str, argv)), cwd=WORKDIR, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    except subprocess.TimeoutExpired:
        save(label + ".status.json", {"returncode": None, "error": "timeout"})
        checkpoint()
        raise
    text = redact(result.stdout)
    (OUT / (label + ".log")).write_text(text)
    save(label + ".status.json", {"returncode": result.returncode})
    checkpoint()
    print(text[-6000:])
    if required and result.returncode:
        raise RuntimeError(label + " 失败，查看对应日志")
    return result

code_files = {}
for folder in ("cachepilot", "benchmarks", "config", "requirements", "workloads", "tests", "notebooks"):
    for path in sorted((WORKDIR / folder).rglob("*")):
        if path.is_file() and path.suffix in {".py", ".json", ".jsonl", ".txt", ".ipynb"}:
            code_files[str(path.relative_to(WORKDIR))] = hashlib.sha256(path.read_bytes()).hexdigest()
for name in ("pyproject.toml", "main.py", ".python-version"):
    code_files[name] = hashlib.sha256((WORKDIR / name).read_bytes()).hexdigest()
save("source.json", {
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "git_dirty": bool(subprocess.check_output(["git", "status", "--porcelain"], text=True)),
    "file_sha256": code_files})
checkpoint()
print("本轮证据:", BACKUP)


Mounted at /content/drive
本轮证据: /content/drive/MyDrive/CachePilot-evidence/20260928T210520-d719e8


## 2. 独立解释器与 GPU 环境门禁

项目固定 Python **3.13.15**。通过 uv 在 /content 安装独立解释器；所有服务/实验使用 PY，不依赖 Colab 内核版本，也无需重启内核。若锁定 Python 或依赖不可下载，结论是环境阻塞；不要用 ignore-requires-python 绕过。只安装 Torch GPU 依赖，不额外安装 vLLM。

要求 Linux x86_64、恰好一张 NVIDIA GPU、至少 8 GiB 显存和 10 GiB 空闲磁盘、CUDA 可用、模型/tokenizer revision 固定。

In [2]:
command([sys.executable, "-m", "pip", "install", "uv"], "bootstrap-uv")
UV = shutil.which("uv")
assert UV
command([UV, "python", "install", "3.13.15"], "python-install")
VENV = Path("/content") / ("cachepilot-venv-" + SESSION)
command([UV, "venv", "--python", "3.13.15", VENV], "venv")
PY = str(VENV / "bin/python")
command([UV, "pip", "install", "--python", PY, "-r", "requirements/requirements-gpu.txt",
         "pytest==8.3.5", "httpx==0.28.1"], "install")
command([PY, "-m", "benchmarks.gpu_environment", "--root", WORKDIR, "--require-torch"], "environment")
command([UV, "pip", "freeze", "--python", PY], "dependencies")
result = command([PY, "-c", "import torch; print('bfloat16' if torch.cuda.is_bf16_supported(including_emulation=False) else 'float16')"], "dtype")
DTYPE = result.stdout.strip().splitlines()[-1]
save("session.json", {"session": SESSION, "dtype": DTYPE, "python": PY, "executor": "TorchExecutor"})


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 124.9 MB/s eta 0:00:00

Installed Python 3.13.15 in 1.36s
 + cpython-3.13.15-linux-x86_64-gnu (python3.13)

Using CPython 3.13.15
Creating virtual environment at: /content/cachepilot-venv-20260928T210520-d719e8
Activate with: source /content/cachepilot-venv-20260928T210520-d719e8/bin/activate

Using Python 3.13.15 environment at: /content/cachepilot-venv-20260928T210520-d719e8
Resolved 64 packages in 607ms
Prepared 64 packages in 1m 03s
Installed 64 packages in 215ms
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + cachepilot==0.1.0 (from file:///content/CachePilot-20260928T210520-d719e8)
 + certifi==2026.7.22
 + click==8.5.0
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.2
 + cuda-toolkit==13.0.2
 + fastapi==0.115.12
 + filelock==4.0.5
 + fsspec==2026.9.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.5.0
 + idna==3.20
 + iniconfig==2.3.0
 + jinja2==3.1.6
 + markd

## 3. Phase 0 前置出口与 API/SSE 回归

必须先通过 CPU 不变量和 API/SSE/取消/超时测试。此处包含替身后端，只证明控制路径；真实 GPU 证据来自后续步骤。

In [3]:
command([PY, "-m", "benchmarks.phase0_exit"], "phase0", timeout=1800)
command([PY, "-m", "pytest", "-q", "tests/integration/test_gateway_api.py",
         "tests/integration/test_vllm_gateway.py", "tests/integration/test_gateway_policies.py",
         "tests/unit/test_torch_executor.py", "tests/unit/test_server.py"],
        "api-regression", timeout=1800)


EXPERIMENT_VALID: /tmp/tmpu9pa4bb9/formal/summary.json
----------------------------------------------------------------------
Ran 193 tests in 2.488s

OK
record-test-0001 200 FINISHED
record-test-0000 200 FINISHED
record-test-0003 200 FINISHED
record-test-0002 200 FINISHED
PHASE0_EXIT=PASS
- CPU tests: PASS
- resource invariants: covered by registry/admission/executor tests
- fixed workload replay: PASS
- logical/physical KV boundary ADR: PASS
- GPU integration gate: CLOSED until this check passes

..............................                                           [100%]
30 passed in 2.18s



CompletedProcess(args=['/content/cachepilot-venv-20260928T210520-d719e8/bin/python', '-m', 'pytest', '-q', 'tests/integration/test_gateway_api.py', 'tests/integration/test_vllm_gateway.py', 'tests/integration/test_gateway_policies.py', 'tests/unit/test_torch_executor.py', 'tests/unit/test_server.py'], returncode=0, stdout='..............................                                           [100%]\n30 passed in 2.18s\n')

## 4. 服务管理与固定协议（第二轮 mixed-policy-v2）

四策略和三次重复保持不变。每轮重启 → 串行 warm-up → 正式测量。T4 使用原生 BF16 支持检测得到的 float16；不复用上一轮 bfloat16 数据。

每次64请求，两tenant，prompt目标1024/512，输出预算1024/128，指令分别要求只回复OK或列出1到12。实际输出必须从usage核实；若仍顶满预算，Adaptive 对照保持 INCONCLUSIVE。

每波8请求、HTTP并发8，等待整波结束再发下一波，避免被拒绝线程立即提交剩余全部请求，重演75%拒绝。Strict可能因逻辑KV不足拒绝部分请求，Adaptive在历史输出短于预算时应减少reservation。Torch只有1个执行槽，其他已准入请求在Scheduler等待，记录调度等待、选择顺序和cache boost。

这是闭环分波负载：trace arrival_ms=0表示可发送，实际到达/完成时间以observations为准；策略时长会改变后续波的实际到达时间。固定输入/seed/分波配置，不宣称严格墙钟开放环回放。P99仍为小样本描述。

In [4]:
BASE_URL = "http://127.0.0.1:8000"
STRATEGIES = [("strict", "fcfs", "blind"), ("adaptive", "fcfs", "blind"),
              ("strict", "wfq", "blind"), ("strict", "wfq", "aware")]
CONTEXTS = [1024, 1024, 512, 512] * 16
REPETITIONS, CONCURRENCY, MAX_TOKENS = 3, 8, 1024
WAVE_SIZE = 8
WORKLOAD_PROFILE = "mixed-policy-v2"
server = None
server_handle = None

def stop_server():
    global server, server_handle
    if server is not None:
        server.terminate()
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait(timeout=10)
        server = None
    if server_handle is not None:
        server_handle.close()
        server_handle = None
    for path in OUT.glob("server-*.log"):
        path.write_text(redact(path.read_text(errors="replace")))
    checkpoint()

atexit.register(stop_server)

def start_server(strategy, label):
    global server, server_handle
    import socket
    stop_server()
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", 8000)) == 0:
            raise RuntimeError("8000 被其他服务占用，请停止旧 notebook 的服务")
    server_handle = (OUT / ("server-" + label + ".log")).open("w")
    server = subprocess.Popen([PY, "main.py", "--host", "127.0.0.1", "--port", "8000",
        "--dtype", DTYPE, "--admission", strategy[0], "--scheduler", strategy[1],
        "--prefix-mode", strategy[2]], cwd=WORKDIR, stdout=server_handle, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 900
    while time.monotonic() < deadline:
        if server.poll() is not None:
            stop_server()
            raise RuntimeError("服务启动失败，检查 server 日志")
        try:
            with urllib.request.urlopen(BASE_URL + "/readyz", timeout=2) as response:
                if json.load(response)["status"] == "ready":
                    return
        except (OSError, ValueError):
            time.sleep(1)
    stop_server()
    raise TimeoutError("加载模型超过 900 秒")

save("protocol.json", {"strategies": STRATEGIES, "contexts": CONTEXTS,
    "repetitions": REPETITIONS, "concurrency": CONCURRENCY, "max_tokens": MAX_TOKENS,
    "warmup_concurrency": 1, "tenants": ["team-a", "team-b"], "seed": 7,
    "arrival_spacing_ms": 0, "dtype": DTYPE, "wave_size": WAVE_SIZE,
    "workload_profile": WORKLOAD_PROFILE, "execution_slots": 1})


## 5. 渐进压测

四种策略分别扫描短到长 prompt、并发1→16，每点三次，全量成功才列为安全点。目标长度是重复词数，实际 tokenizer 长度从 telemetry 读取；8192 加模板/输出可能先被上下文限制拒绝。429 是过载保护；普通上下文拒绝不是 OOM。观察到 OOM 后停止升压。

报告按 (实际上下文, 并发) 点解释，禁止把不同点的两个最大值拼成安全配置；未见 OOM 只代表本次未触及，不能推断最大显存容量。

In [5]:
for strategy in STRATEGIES:
    name = "-".join(strategy)
    start_server(strategy, "progressive-" + name)
    try:
        target = OUT / "progressive" / name
        command([PY, "-m", "benchmarks.progressive_load", "--base-url", BASE_URL,
            "--tenant-id", "team-a", "--output-dir", target,
            "--contexts", "32,128,512,2048,4096,8192", "--concurrencies", "1,2,4,8,16",
            "--max-tokens", "16", "--repetitions", "3", "--warmup"],
            "progressive-" + name, timeout=14400)
        report = json.loads((target / "progressive_report.json").read_text())
        print(name, json.dumps(report["safety"], ensure_ascii=False))
    finally:
        stop_server()


 safe=True counts={'success': 4}
context=128 concurrency=4 repetition=2 safe=True counts={'success': 4}
context=128 concurrency=4 repetition=3 safe=True counts={'success': 4}
context=128 concurrency=8 repetition=1 safe=True counts={'success': 8}
context=128 concurrency=8 repetition=2 safe=True counts={'success': 8}
context=128 concurrency=8 repetition=3 safe=True counts={'success': 8}
context=128 concurrency=16 repetition=1 safe=False counts={'success': 8, 'overload_protected': 8}
context=128 concurrency=16 repetition=2 safe=False counts={'success': 8, 'overload_protected': 8}
context=128 concurrency=16 repetition=3 safe=False counts={'success': 8, 'overload_protected': 8}
context=512 concurrency=1 repetition=1 safe=True counts={'success': 1}
context=512 concurrency=1 repetition=2 safe=True counts={'success': 1}
context=512 concurrency=1 repetition=3 safe=True counts={'success': 1}
context=512 concurrency=2 repetition=1 safe=True counts={'success': 2}
context=512 concurrency=2 repetiti

## 6. 正式矩阵：生成原始四件套

实际生成 manifest/trace/requests/summary，不改名 progressive 文件。额外保存 observations.json：原始 query/ledger、实际发送时间、完成时间和测量窗口吞吐。trace_id 含规范化 trace 哈希，内容变化即比较失败。

禁止覆盖旧 run；中断后保留半成品，开新会话重跑。warm-up 不参加性能统计。

In [6]:
formal_runs, measured_runs = [], []
for repetition in range(REPETITIONS):
    offset = repetition % len(STRATEGIES)
    for strategy in STRATEGIES[offset:] + STRATEGIES[:offset]:
        label = "-".join(strategy) + "-" + str(repetition)
        start_server(strategy, label)
        try:
            for warmup in (True, False):
                name = label + ("-warmup" if warmup else "-measured")
                target = OUT / "matrix" / name
                argv = [PY, "-m", "benchmarks.formal_gpu_run", "--root", WORKDIR,
                    "--base-url", BASE_URL, "--output", target, "--run-id", name,
                    "--admission", strategy[0], "--scheduler", strategy[1],
                    "--prefix-mode", strategy[2], "--dtype", DTYPE,
                    "--repetition-index", str(repetition), "--seed", "7",
                    "--tenants", "team-a", "team-b", "--max-tokens", str(MAX_TOKENS),
                    "--concurrency", "1" if warmup else str(CONCURRENCY),
                    "--wave-size", str(WAVE_SIZE), "--workload-profile", WORKLOAD_PROFILE,
                    "--contexts", *map(str, CONTEXTS)]
                if warmup:
                    argv.append("--warmup")
                command(argv, name)
                formal_runs.append(target)
                if not warmup:
                    measured_runs.append(target)
        finally:
            stop_server()
save("run-index.json", {"formal": [str(p.relative_to(OUT)) for p in formal_runs],
                       "measured": [str(p.relative_to(OUT)) for p in measured_runs]})


EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-fcfs-blind-0-warmup/summary.json
strict-fcfs-blind-0-warmup-0000 200 FINISHED
strict-fcfs-blind-0-warmup-0001 200 FINISHED
strict-fcfs-blind-0-warmup-0002 200 FINISHED
strict-fcfs-blind-0-warmup-0003 200 FINISHED
strict-fcfs-blind-0-warmup-0004 200 FINISHED
strict-fcfs-blind-0-warmup-0005 200 FINISHED
strict-fcfs-blind-0-warmup-0006 200 FINISHED
strict-fcfs-blind-0-warmup-0007 200 FINISHED
strict-fcfs-blind-0-warmup-0008 200 FINISHED
strict-fcfs-blind-0-warmup-0009 200 FINISHED
strict-fcfs-blind-0-warmup-0010 200 FINISHED
strict-fcfs-blind-0-warmup-0011 200 FINISHED
strict-fcfs-blind-0-warmup-0012 200 FINISHED
strict-fcfs-blind-0-warmup-0013 200 FINISHED
strict-fcfs-blind-0-warmup-0014 200 FINISHED
strict-fcfs-blind-0-warmup-0015 200 FINISHED
strict-fcfs-blind-0-warmup-0016 200 FINISHED
strict-fcfs-blind-0-warmup-0017 200 FINISHED
strict-fcfs-blind-0-warmup-0018 200 FINISHED
strict-fcfs-blind-0-warmup-001

## 7. 对照门禁与逐次结果

重新分析原始数据并检查版本/trace与重复次数。新增policy证据：Adaptive非fallback和预留缩减请求数、Scheduler等待P95、逻辑命中及实际cache boost次数；query.adaptive保留历史量/估计误差，query.policy不包含prompt或token IDs。

MATRIX_VALID只表示格式和基本变量可比。最终还要求每次至少24个完成请求、拒绝率≤50%、Scheduler等待P95≥1ms、每轮Adaptive学习并缩减预留、每轮prefix-aware有命中和dispatch boost；未触发仍标 INCONCLUSIVE。

逻辑prefix仅验证调度偏好，不宣称Torch物理KV复用或必然吞吐提升。保留实际窗口吞吐、全部重复及完整输入。

In [7]:
for path in formal_runs:
    command([PY, "-m", "benchmarks.analyze", "--manifest", path / "manifest.json",
        "--trace", path / "trace.jsonl", "--requests", path / "requests.jsonl",
        "--output", path / "summary.json"], "analyze-" + path.name)
command([PY, "-m", "benchmarks.strategy_matrix", "--metric", "ttft_ms",
         "--output", OUT / "matrix.json", *formal_runs], "matrix-validation")
rows = []
for path in measured_runs:
    summary = json.loads((path / "summary.json").read_text())
    observation = json.loads((path / "observations.json").read_text())
    requests = observation["requests"]
    policies = [request["query"].get("policy", {}) for request in requests]
    waits = sorted(policy["scheduler_wait_ms"] for policy in policies
                   if policy.get("scheduler_wait_ms") is not None)
    rows.append({
        "run": path.name, "strategy": path.name.rsplit("-", 2)[0],
        "ttft_p95_ms": summary["metrics"]["ttft_ms"]["p95"],
        "ttft_p99_ms": summary["metrics"]["ttft_ms"]["p99"],
        "tpot_p95_ms": summary["metrics"]["tpot_ms"]["p95"],
        "total_p99_ms": summary["metrics"]["total_ms"]["p99"],
        "wall_tokens_per_s": observation["wall_throughput_tokens_per_s"],
        "fairness_jain": summary["fairness_jain"], "rejection_rate": summary["rejection_rate"],
        "request_kv_peak_blocks": summary["resource_peaks"]["reserved_blocks_peak"],
        "logical_hits": summary["prefix_observation"]["logical_hits"],
        "cache_boosts": sum(policy.get("cache_boosted", False) for policy in policies),
        "scheduler_wait_p95_ms": waits[(95 * len(waits) + 99) // 100 - 1] if waits else None,
        "adaptive_learned": sum(policy.get("admission") == "adaptive"
                                and policy.get("fallback_to_strict") is False for policy in policies),
        "reduced_reservations": sum(
            request["query"].get("policy", {}).get("estimated_output_tokens") is not None
            and request["query"]["policy"]["estimated_output_tokens"] < request["input"]["max_tokens"]
            for request in requests),
        "finished": summary["terminal_counts"].get("FINISHED", 0),
        "failed": summary["terminal_counts"].get("FAILED", 0),
    })
save("comparison-rows.json", rows)
from IPython.display import display, Markdown
import pandas as pd
display(pd.DataFrame(rows))


EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-fcfs-blind-0-warmup/summary.json

EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-fcfs-blind-0-measured/summary.json

EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/adaptive-fcfs-blind-0-warmup/summary.json

EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/adaptive-fcfs-blind-0-measured/summary.json

EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-wfq-blind-0-warmup/summary.json

EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-wfq-blind-0-measured/summary.json

EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-wfq-aware-0-warmup/summary.json

EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-wfq-aware-0-measured/summary.json

EXPERIMENT_VALID: /content/cachepilot-phase1/20260928T210520-d71

,run,strategy,ttft_p95_ms,ttft_p99_ms,tpot_p95_ms,total_p99_ms,wall_tokens_per_s,fairness_jain,rejection_rate,request_kv_peak_blocks,logical_hits,cache_boosts,scheduler_wait_p95_ms,adaptive_learned,reduced_reservations,finished,failed
0,strict-fcfs-blind-0-measured,strict-fcfs-blind,1079.598808,1116.977892,0.050834,4199.278786,35.725008,0.992675,0.312500,131,44,0,3176.968482,0,0,44,0
1,adaptive-fcfs-blind-0-measured,adaptive-fcfs-blind,1042.529044,1082.321470,0.065322,4336.706114,35.639164,1.000000,0.000000,70,64,0,3301.643494,64,64,64,0
2,strict-wfq-blind-0-measured,strict-wfq-blind,1050.459101,1118.804008,0.050910,4164.152851,36.082874,0.994828,0.281250,131,46,0,3233.120328,0,0,46,0
3,strict-wfq-aware-0-measured,strict-wfq-aware,1165.316046,1192.252085,0.049867,4443.359770,34.932446,0.993877,0.296875,131,45,1,4283.035200,0,0,45,0
4,adaptive-fcfs-blind-1-measured,adaptive-fcfs-blind,1093.540737,1181.751353,0.063437,4492.672953,34.405825,1.000000,0.000000,70,64,0,3554.881264,64,64,64,0
5,strict-wfq-blind-1-measured,strict-wfq-blind,1108.238213,1171.009878,0.048969,4384.928474,34.875443,0.999995,0.312500,131,44,0,3268.614319,0,0,44,0
6,strict-wfq-aware-1-measured,strict-wfq-aware,1134.135779,1194.412630,0.050531,4539.662644,35.332838,0.999993,0.265625,131,47,1,4127.984460,0,0,47,0
7,strict-fcfs-blind-1-measured,strict-fcfs-blind,1054.212326,1108.810701,0.051606,4247.391086,34.866407,0.991997,0.328125,131,43,0,3177.610946,0,0,43,0
8,strict-wfq-blind-2-measured,strict-wfq-blind,1104.267547,1189.713482,0.050527,4435.404066,33.759290,0.962695,0.343750,131,42,0,3226.491393,0,0,42,0
9,strict-wfq-aware-2-measured,strict-wfq-aware,1156.404417,1200.306868,0.050007,4591.945871,34.863564,0.992415,0.328125,131,43,2,3527.861334,0,0,43,0


## 8. 五项故障验证：控制路径与真实 GPU

先运行原有确定性检查，再用独立真实模型进程验证：HTTP取消/重复取消、真实SSE连接关闭、短deadline、model.forward异常、CUDA分配OOM。逐项检查唯一终态、逻辑reservation/active回零、生成线程结束、allocated显存回到基线（容差16MiB）、正常请求恢复。

本单元会**主动触发一次真实 CUDA OOM**：在 model.forward 内申请超过整卡容量的 tensor，必须由 CUDA allocator 抛出 OutOfMemoryError。它验证受控分配失败后的回收，**不是自然负载容量边界或worker进程崩溃**。自然OOM是否出现由渐进扫描回答。异常同样明确标注为受控注入。只影响当前Colab实验进程。

任一项失败都写出 FAIL；若runtime被平台终止，保留Drive已有证据，记为未完成。

In [8]:
stop_server()
command([PY, "-m", "benchmarks.chaos_validation", "--output", OUT / "chaos-control.json"], "chaos-control")
command([PY, "-m", "benchmarks.colab_phase1", "gpu-chaos",
         "--dtype", DTYPE, "--output", OUT / "chaos-gpu.json"],
        "chaos-gpu", timeout=1800, required=False)
print((OUT / "chaos-gpu.json").read_text() if (OUT / "chaos-gpu.json").exists()
      else "缺少 GPU 故障报告，Phase 1 不得通过")


CHAOS_VALIDATION=PASS: /content/cachepilot-phase1/20260928T210520-d719e8/chaos-control.json

`torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5863.08it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/CachePilot-20260928T210520-d719e8/benchmarks/colab_phase1.py", line 355, in <module>
    main()
    ~~~~^^
  File "/content/CachePilot-20260928T210520-d719e8/benchmarks/colab_phase1.py", line 351, in main
    gpu_chaos(args.output, args.dtype)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/CachePilot-20260928T210520-d719e8/benchmarks/colab_phase1.py", line 322, in gpu_chaos
    assert case["status"] == "PASS", (
           ^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: d

## 9. Sim 原始运行与 Phase 1 子门禁

真实执行 SimExecutor，保留完整事件与四件套；标记 simulated，不与GPU时延/吞吐混合比较。Sim的GPU秒和成本为null。

现有 phase1_exit.py 仅是子门禁；最终结论还必须结合渐进压测、矩阵、真实故障证据和策略有效性。

In [9]:
command([PY, "-m", "benchmarks.colab_phase1", "sim", "--reference", measured_runs[0],
         "--output", OUT / "sim"], "sim")
command([PY, "-m", "benchmarks.phase1_exit", "--chaos-report", OUT / "chaos-gpu.json",
         "--output", OUT / "phase1-exit.json", OUT / "sim", *measured_runs],
        "phase1-gate", timeout=1800, required=False)



.........
----------------------------------------------------------------------
Ran 9 tests in 0.068s

OK
PHASE1_EXIT=FAIL: chaos report must pass cancellation, disconnect, timeout, exception and OOM



CompletedProcess(args=['/content/cachepilot-venv-20260928T210520-d719e8/bin/python', '-m', 'benchmarks.phase1_exit', '--chaos-report', '/content/cachepilot-phase1/20260928T210520-d719e8/chaos-gpu.json', '--output', '/content/cachepilot-phase1/20260928T210520-d719e8/phase1-exit.json', '/content/cachepilot-phase1/20260928T210520-d719e8/sim', '/content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-fcfs-blind-0-measured', '/content/cachepilot-phase1/20260928T210520-d719e8/matrix/adaptive-fcfs-blind-0-measured', '/content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-wfq-blind-0-measured', '/content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-wfq-aware-0-measured', '/content/cachepilot-phase1/20260928T210520-d719e8/matrix/adaptive-fcfs-blind-1-measured', '/content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-wfq-blind-1-measured', '/content/cachepilot-phase1/20260928T210520-d719e8/matrix/strict-wfq-aware-1-measured', '/content/cachepilot-phase1/2026092

## 10. 产出四项结论

PASS表示原始证据、基本矩阵门禁和策略触发覆盖通过，不代表策略一定优于基线或性能SLO达标。检查policy-coverage.json定位 INCONCLUSIVE：无命中/boost、Adaptive始终fallback或未缩减、队列竞争不足、样本不足均不能通过。

保留无收益或负向结果。逻辑KV为逐请求reservation峰值。Torch TTFT/TPOT仍是整段生成后的Gateway交付语义；scheduler_wait_p95_ms单独展示执行槽等待。不得将它们解释为逐token GPU decode 或物理prefix收益。

In [10]:
def read_optional(name):
    path = OUT / name
    return json.loads(path.read_text()) if path.exists() else {}

def succeeded(label):
    return read_optional(label + ".status.json").get("returncode") == 0

safe_points = {}
for strategy in STRATEGIES:
    name = "-".join(strategy)
    report = read_optional("progressive/" + name + "/progressive_report.json")
    safe_points[name] = [point for point in report.get("matrix_points", []) if point["safe"]]
progressive_ok = all(safe_points.values()) and all(succeeded("progressive-" + "-".join(s)) for s in STRATEGIES)
matrix_ok = succeeded("matrix-validation") and len(rows) == 4 * REPETITIONS
prefix_hits = sum(row["logical_hits"] for row in rows if row["strategy"] == "strict-wfq-aware")
policy_coverage = {
    "enough_completions": all(row.get("finished", 0) >= 24 for row in rows),
    "bounded_rejection": all(row["rejection_rate"] <= 0.5 for row in rows),
    "scheduler_queue_exercised": all((row.get("scheduler_wait_p95_ms") or 0) >= 1 for row in rows),
    "adaptive_learned_and_reduced": all(
        row.get("adaptive_learned", 0) > 0 and row.get("reduced_reservations", 0) > 0
        for row in rows if row["strategy"] == "adaptive-fcfs-blind"),
    "prefix_hits_and_dispatch_boosts": all(
        row["logical_hits"] > 0 and row.get("cache_boosts", 0) > 0
        for row in rows if row["strategy"] == "strict-wfq-aware"),
}
save("policy-coverage.json", policy_coverage)
matrix_status = ("PASS" if all(policy_coverage.values()) else "INCONCLUSIVE") if matrix_ok and not sum(row["failed"] for row in rows) else "FAIL"
chaos = read_optional("chaos-gpu.json")
chaos_ok = (succeeded("chaos-control") and succeeded("chaos-gpu") and chaos.get("status") == "PASS"
    and {case["name"] for case in chaos.get("cases", [])} == {"cancel", "disconnect", "timeout", "exception", "oom"})
exit_ok = (progressive_ok and matrix_status == "PASS" and chaos_ok and succeeded("phase0")
           and succeeded("api-regression") and succeeded("phase1-gate"))
conclusion = {
    "session": SESSION, "渐进压测": "PASS" if progressive_ok else "FAIL",
    "必要对照": matrix_status, "故障验证": "PASS" if chaos_ok else "FAIL",
    "Phase1出口": "PASS" if exit_ok else "NOT_PASSED",
    "safe_points_by_strategy": safe_points, "prefix_aware_logical_hits": prefix_hits,
    "policy_coverage": policy_coverage,
    "limits": [
        "Torch TTFT/TPOT 是整段生成后的 Gateway 交付指标，非逐 token GPU decode 性能",
        "物理 prefix 命中不可观测；无逻辑命中证据时不能声称 prefix 调度收益",
        "OOM 是真实 CUDA allocator 受控注入，自然容量边界见 progressive",
        "成本为估算；未配置 GPU 小时价格时 estimated_cost=null",
        "static vs continuous: N/A；当前 Torch 不支持两种 batching",
        "小样本 P99、同权 tenant token Jain 指数仅描述本固定 workload",
    ]}
save("conclusions.json", conclusion)
text = "# Colab Phase 1 验证结论\n\n"
text += "\n".join("- " + key + ": " + conclusion[key] for key in ("渐进压测", "必要对照", "故障验证", "Phase1出口"))
text += "\n\n## 三组必要对照：正式重复的中位数\n\n"
from statistics import median
pairs = [("Strict vs Adaptive", "strict-fcfs-blind", "adaptive-fcfs-blind"),
         ("FCFS vs WFQ", "strict-fcfs-blind", "strict-wfq-blind"),
         ("prefix-blind vs prefix-aware", "strict-wfq-blind", "strict-wfq-aware")]
for label, left, right in pairs:
    text += "### " + label + "\n\n"
    for field in ("ttft_p95_ms", "tpot_p95_ms", "total_p99_ms", "wall_tokens_per_s",
                  "fairness_jain", "rejection_rate", "request_kv_peak_blocks",
                  "scheduler_wait_p95_ms", "cache_boosts", "reduced_reservations"):
        groups = [[row[field] for row in rows if row["strategy"] == strategy and row.get(field) is not None]
                  for strategy in (left, right)]
        values = [median(group) if group else None for group in groups]
        text += "- " + field + ": " + str(values[0]) + " → " + str(values[1]) + "\n"
    text += "\n"
text += "## 策略触发覆盖\n\n" + "\n".join("- " + key + ": " + str(value) for key, value in policy_coverage.items()) + "\n\n"
if not prefix_hits:
    text += "prefix-aware 无逻辑命中证据：必要对照结论不足，检查 Gateway → PrefixIndex → Scheduler 接线。\n\n"
text += "## 结论边界\n\n" + "\n".join("- " + limit for limit in conclusion["limits"])
text += "\n\n安全点见 conclusions.json；逐次结果见 comparison-rows.json；原始请求见 matrix/。\n"
(OUT / "report.md").write_text(text)
checkpoint()
display(Markdown(text))


# Colab Phase 1 验证结论

- 渐进压测: PASS
- 必要对照: PASS
- 故障验证: FAIL
- Phase1出口: NOT_PASSED

## 三组必要对照：正式重复的中位数

### Strict vs Adaptive

- ttft_p95_ms: 1054.212326 → 1066.6377730000002
- tpot_p95_ms: 0.050834055555555556 → 0.06493436111111112
- total_p99_ms: 4199.278786 → 4492.672952999999
- wall_tokens_per_s: 35.26635800369344 → 34.74228310604001
- fairness_jain: 0.9919968698868891 → 1.0
- rejection_rate: 0.328125 → 0.0
- request_kv_peak_blocks: 131 → 70
- scheduler_wait_p95_ms: 3176.968482 → 3554.881264
- cache_boosts: 0 → 0
- reduced_reservations: 0 → 64

### FCFS vs WFQ

- ttft_p95_ms: 1054.212326 → 1104.2675470000001
- tpot_p95_ms: 0.050834055555555556 → 0.05052661111111111
- total_p99_ms: 4199.278786 → 4384.928473999999
- wall_tokens_per_s: 35.26635800369344 → 34.875443468284615
- fairness_jain: 0.9919968698868891 → 0.9948275800310202
- rejection_rate: 0.328125 → 0.3125
- request_kv_peak_blocks: 131 → 131
- scheduler_wait_p95_ms: 3176.968482 → 3233.120328
- cache_boosts: 0 → 0
- reduced_reservations: 0 → 0

### prefix-blind vs prefix-aware

- ttft_p95_ms: 1104.2675470000001 → 1156.404417
- tpot_p95_ms: 0.05052661111111111 → 0.050007027777777775
- total_p99_ms: 4384.928473999999 → 4539.662644
- wall_tokens_per_s: 34.875443468284615 → 34.93244592363558
- fairness_jain: 0.9948275800310202 → 0.9938768344194404
- rejection_rate: 0.3125 → 0.296875
- request_kv_peak_blocks: 131 → 131
- scheduler_wait_p95_ms: 3233.120328 → 4127.98446
- cache_boosts: 0 → 1
- reduced_reservations: 0 → 0

## 策略触发覆盖

- enough_completions: True
- bounded_rejection: True
- scheduler_queue_exercised: True
- adaptive_learned_and_reduced: True
- prefix_hits_and_dispatch_boosts: True

## 结论边界

- Torch TTFT/TPOT 是整段生成后的 Gateway 交付指标，非逐 token GPU decode 性能
- 物理 prefix 命中不可观测；无逻辑命中证据时不能声称 prefix 调度收益
- OOM 是真实 CUDA allocator 受控注入，自然容量边界见 progressive
- 成本为估算；未配置 GPU 小时价格时 estimated_cost=null
- static vs continuous: N/A；当前 Torch 不支持两种 batching
- 小样本 P99、同权 tenant token Jain 指数仅描述本固定 workload

安全点见 conclusions.json；逐次结果见 comparison-rows.json；原始请求见 matrix/。


## 11. 归档下载（失败也执行）

保存代码快照、环境/依赖、每个原始run、故障事件、日志、checksums和结论。前面中断则写出 NOT_COMPLETED，绝不沿用旧PASS。下载ZIP后解压到独立runs目录，依据证据更新TODO并链接验收记录。

排查：环境失败检查锁定版本可用性/GPU/空间；启动失败查看server日志/T4 dtype/端口；429是准入保护，不是OOM；分析失败检查原始字段/trace/版本，禁止手改summary过门禁；故障失败查看终态/events/显存/recovery，失败就是本次实验结论。

In [11]:
if "stop_server" in globals():
    stop_server()
if not (OUT / "conclusions.json").exists():
    save("conclusions.json", {"session": SESSION, "Phase1出口": "NOT_COMPLETED",
                             "reason": "前置步骤中断，检查 status.json 和日志"})
    (OUT / "report.md").write_text("# Phase 1 未完成\n\n前置步骤中断，参见日志；不能勾选 TODO。\n")
for relative in code_files:
    target = OUT / "source-snapshot" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(WORKDIR / relative, target)
save("checksums.json", {str(path.relative_to(OUT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(OUT.rglob("*")) if path.is_file() and path.name != "checksums.json"})
checkpoint()
archive = shutil.make_archive(str(OUT.parent / (SESSION + "-evidence")), "zip",
                             root_dir=OUT.parent, base_dir=OUT.name)
saved_archive = BACKUP.parent / Path(archive).name
shutil.copy2(archive, saved_archive)
print("证据 ZIP:", saved_archive)
print("结论:", BACKUP / "report.md")
from google.colab import files
files.download(archive)


证据 ZIP: /content/drive/MyDrive/CachePilot-evidence/20260928T210520-d719e8-evidence.zip
结论: /content/drive/MyDrive/CachePilot-evidence/20260928T210520-d719e8/report.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>